# Projeto Final — Redes Neurais Artificiais Aplicadas ao Setor Financeiro
## Previsão de Inadimplência em Cartão de Crédito com Redes Neurais Tabulares (PyTorch)

**Integrantes:** [Nome 1], [Nome 2], [Nome 3], [Nome 4]  
**Data de Apresentação:** 23 de novembro de 2026

---

## Etapa 1 — Definição e Compreensão do Problema

### 1.1 Contexto do Problema
No setor financeiro e de meios de pagamento, a concessão de crédito rotativo (como cartões de crédito) envolve um equilíbrio constante entre **expansão da carteira** e **controle de risco de inadimplência (*default*)**. Decisões imprecisas geram dois tipos de prejuízo assimétricos:
- **Falso Negativo (FN):** Aprovar ou manter limite para um cliente que ficará inadimplente, gerando perda direta do principal não pago e custos de cobrança.
- **Falso Positivo (FP):** Recusar ou restringir crédito de um bom pagador, gerando custo de oportunidade (perda de receita de juros, anuidade e taxas de intercâmbio).

### 1.2 Objetivo da Solução
Desenvolver, treinar e avaliar modelos baseados em **Redes Neurais Artificiais (PyTorch)** capazes de estimar a **probabilidade de inadimplência no mês seguinte** a partir do histórico financeiro, faturas, pagamentos e perfil demográfico do cliente, além de calibrar o limiar de decisão (*threshold*) sob uma ótica de custo financeiro.

### 1.3 Tipo de Tarefa de Machine Learning
- **Aprendizado Supervisionado — Classificação Binária Probabilística** (`0` = adimplente, `1` = inadimplente no próximo mês).

### 1.4 Apresentação e Justificativa do Dataset
- **Dataset:** *Default of Credit Card Clients* (Yeh & Lien, 2009 — UCI Machine Learning Repository, ID 350).
- **Dimensão:** 30.000 clientes e 23 variáveis explicativas (demográficas, limite de crédito e série histórica de 6 meses de status de pagamento, valor de fatura e valor pago).
- **Justificativa:**
  1. É um dataset público de referência na literatura de risco de crédito (*Expert Systems with Applications*, 2009).
  2. Apresenta **desafios reais de dados financeiros**: desbalanceamento moderado (~22,1% de inadimplência), variáveis assimétricas (*heavy-tailed*) e códigos categóricos não documentados que exigem análise exploratória e pré-processamento criterioso.
  3. Possui histórico temporal de 6 meses (`t-1` a `t-6`), permitindo *feature engineering* financeiro autoral (utilização de limite, razão pagamento/fatura e tendência de atraso).

In [ ]:
# Configuração de ambiente e bibliotecas
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    roc_curve, precision_recall_curve, classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Reprodutibilidade
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_theme(style='whitegrid', context='notebook')
print(f'PyTorch: {torch.__version__} | Dispositivo: {device}')

In [ ]:
# Carregamento do dataset público (UCI direto com fallback para OpenML)
CANONICAL_COLS = [
    'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

def load_credit_default_dataset():
    uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls'
    try:
        df_raw = pd.read_excel(uci_url, header=1)
        if 'ID' in df_raw.columns:
            df_raw = df_raw.drop(columns=['ID'])
        df_raw = df_raw.rename(columns={'default payment next month': 'TARGET'})
        print('Dataset carregado diretamente do repositório UCI.')
        return df_raw
    except Exception as e:
        print(f'Fallback para OpenML (motivo UCI: {e})...')
        data = fetch_openml(data_id=42477, as_frame=True, parser='auto')
        df_raw = data.frame.copy()
        # Padroniza nomes de colunas caso OpenML retorne x1..x23
        if 'x1' in df_raw.columns:
            rename_map = {f'x{i+1}': col for i, col in enumerate(CANONICAL_COLS)}
            rename_map['y'] = 'TARGET'
            df_raw = df_raw.rename(columns=rename_map)
        elif 'default payment next month' in df_raw.columns:
            df_raw = df_raw.rename(columns={'default payment next month': 'TARGET'})
        for col in df_raw.columns:
            df_raw[col] = pd.to_numeric(df_raw[col])
        print('Dataset carregado via OpenML (ID 42477).')
        return df_raw

df = load_credit_default_dataset()
print(f'Formato do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head()

---
## Etapa 2 — Análise Exploratória de Dados (EDA)

Nesta etapa investigamos:
1. Tipos de dados, valores ausentes e registros duplicados.
2. Distribuição da variável alvo (`TARGET`) para quantificar o desbalanceamento.
3. **Inconsistências categóricas** conhecidas em `EDUCATION` e `MARRIAGE`.
4. **Assimetria e outliers** em variáveis monetárias (`LIMIT_BAL`, `BILL_AMT*`, `PAY_AMT*`).
5. **Relações entre variáveis** (atraso recente `PAY_0`, limite de crédito e correlações).

In [ ]:
# 2.1 Visão geral, ausentes e duplicatas
print('Valores ausentes totais:', df.isna().sum().sum())
print('Linhas duplicadas:', df.duplicated().sum())
print('\nDistribuição do TARGET:')
target_counts = df['TARGET'].value_counts()
target_pct = df['TARGET'].value_counts(normalize=True) * 100
display(pd.DataFrame({'Quantidade': target_counts, 'Percentual (%)': target_pct.round(2)}))

# Verificação de inconsistências em variáveis categóricas
print('\nValores únicos em SEX (esperado 1, 2):', sorted(df['SEX'].unique()))
print('Valores únicos em EDUCATION (documentado 1..4):', sorted(df['EDUCATION'].unique()))
print('Valores únicos em MARRIAGE (documentado 1..3):', sorted(df['MARRIAGE'].unique()))
print('Valores únicos em PAY_0:', sorted(df['PAY_0'].unique()))

In [ ]:
# 2.2 Estatísticas descritivas das variáveis principais
df[['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1', 'PAY_0']].describe().T[['mean', 'std', 'min', '50%', 'max']]

In [ ]:
# 2.3 Visualizações Exploratórias
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) Desbalanceamento da classe alvo
sns.countplot(data=df, x='TARGET', ax=axes[0, 0], palette=['#2b6cb0', '#e53e3e'])
axes[0, 0].set_title('Distribuição da Variável Alvo (0=Adimplente, 1=Inadimplente)')
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2., p.get_height()),
                        ha='center', va='bottom', fontsize=10)

# 2) Taxa de inadimplência por status de pagamento recente (PAY_0)
pay0_default = df.groupby('PAY_0')['TARGET'].agg(['mean', 'count']).reset_index()
sns.barplot(data=pay0_default, x='PAY_0', y='mean', ax=axes[0, 1], color='#dd6b20')
axes[0, 1].set_title('Taxa de Inadimplência por Status de Pagamento (PAY_0)')
axes[0, 1].set_ylabel('Proporção de Inadimplentes')

# 3) Distribuição de LIMIT_BAL por classe (escala log/boxplot)
sns.boxplot(data=df, x='TARGET', y='LIMIT_BAL', ax=axes[1, 0], palette=['#2b6cb0', '#e53e3e'])
axes[1, 0].set_title('Limite de Crédito (LIMIT_BAL) por Classe')

# 4) Assimetria e outliers em PAY_AMT1 (escala log1p)
sns.histplot(np.log1p(df['PAY_AMT1'].clip(lower=0)), bins=40, kde=True, ax=axes[1, 1], color='#319795')
axes[1, 1].set_title('Distribuição de log1p(PAY_AMT1) — Forte Assimetria')

plt.tight_layout()
plt.show()

In [ ]:
# 2.4 Matriz de correlação das variáveis principais + histórico de atrasos
corr_cols = ['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'BILL_AMT1', 'PAY_AMT1', 'TARGET']
plt.figure(figsize=(10, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Matriz de Correlação (Variáveis Selecionadas)')
plt.tight_layout()
plt.show()

### Principais achados da EDA
1. **Desbalanceamento:** 22,12% dos clientes são inadimplentes (`TARGET=1`), o que torna a **acurácia isolada enganosa** (um modelo trivial que prevê sempre `0` teria 77,88% de acurácia). Por isso usaremos **ROC-AUC, PR-AUC, Recall e F1-Score**, além de ponderação de perda (`pos_weight`).
2. **Inconsistências categóricas:** `EDUCATION` possui códigos `0, 5, 6` não documentados e `MARRIAGE` possui código `0`. Ambos devem ser agrupados na categoria `Outros`.
3. **Poder preditivo de atrasos recentes:** Clientes com `PAY_0 >= 2` (2 ou mais meses de atraso) têm taxa de inadimplência superior a 65%, contra ~10%–17% nos demais.
4. **Assimetria e Outliers:** `BILL_AMT*` (incluindo faturas negativas por estorno/crédito) e `PAY_AMT*` têm caudas longas e outliers extremos, justificando uso de `RobustScaler` e *feature engineering* baseado em razões relativas ao limite.

---
## Etapa 3 — Pré-processamento e Engenharia de Atributos

### Decisões técnicas e justificativas
1. **Remoção de duplicatas exatas** (35 registros) para evitar viés e vazamento entre treino/validação/teste.
2. **Saneamento categórico:** Agrupamento de `EDUCATION` (`0, 5, 6 -> 4`) e `MARRIAGE` (`0 -> 3`) conforme dicionário oficial.
3. **Feature Engineering Financeiro (domínio do problema):**
   - `UTIL_RATIO_1` e `UTIL_RATIO_MEAN`: razão entre fatura e limite de crédito (`BILL_AMT / LIMIT_BAL`). Captura alavancagem/sufocamento financeiro independentemente do tamanho absoluto do limite.
   - `PAY_TO_BILL_1`: razão entre o pagamento realizado e a fatura anterior (`PAY_AMT1 / (BILL_AMT2.clip(0) + 1)`), indicando se o cliente quita integralmente ou rola a dívida.
   - `MAX_DELAY`, `NUM_DELAYS` e `PAY_TREND` (`PAY_0 - PAY_6`): capturam severidade, frequência e piora/melhora recente no comportamento de pagamento.
4. **Divisão Estratificada (70% Treino / 15% Validação / 15% Teste):** Mantém a proporção de inadimplentes (~22,1%) nos três conjuntos.
5. **Prevenção estrita de *Data Leakage*:** O `ColumnTransformer` (`RobustScaler` para numéricas com outliers + `OneHotEncoder` para categóricas nominais) é ajustado (`fit`) **exclusivamente nos dados de treino** e apenas aplicado (`transform`) na validação e no teste.

In [ ]:
df_clean = df.drop_duplicates().copy()
print(f'Registros após remoção de duplicatas: {len(df_clean)} (removidos: {len(df) - len(df_clean)})')

# Saneamento de códigos não documentados
df_clean['EDUCATION'] = df_clean['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df_clean['MARRIAGE'] = df_clean['MARRIAGE'].replace({0: 3})

# Feature Engineering Financeiro
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_amt_cols = [f'PAY_AMT{i}' for i in range(1, 7)]
pay_stat_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

df_clean['UTIL_RATIO_1'] = df_clean['BILL_AMT1'] / (df_clean['LIMIT_BAL'] + 1.0)
df_clean['UTIL_RATIO_MEAN'] = df_clean[bill_cols].mean(axis=1) / (df_clean['LIMIT_BAL'] + 1.0)
df_clean['PAY_TO_BILL_1'] = np.log1p(df_clean['PAY_AMT1'].clip(lower=0)) - np.log1p(df_clean['BILL_AMT2'].clip(lower=0))
df_clean['TOTAL_PAY_6M'] = np.log1p(df_clean[pay_amt_cols].clip(lower=0).sum(axis=1))
df_clean['MAX_DELAY'] = df_clean[pay_stat_cols].max(axis=1)
df_clean['NUM_DELAYS'] = (df_clean[pay_stat_cols] > 0).sum(axis=1)
df_clean['PAY_TREND'] = df_clean['PAY_0'] - df_clean['PAY_6']

X = df_clean.drop(columns=['TARGET'])
y = df_clean['TARGET'].astype(np.int64)

# Divisão estratificada: 70% treino, 15% validação, 15% teste
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print(f'Treino: {X_train.shape} | Validação: {X_val.shape} | Teste: {X_test.shape}')
print(f'Taxa de inadimplência -> Treino: {y_train.mean():.4f} | Val: {y_val.mean():.4f} | Teste: {y_test.mean():.4f}')

In [ ]:
# Pipeline de transformação (fit apenas no treino!)
cat_features = ['SEX', 'EDUCATION', 'MARRIAGE']
num_features = [c for c in X_train.columns if c not in cat_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
    ]
)

X_train_proc = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_proc = preprocessor.transform(X_val).astype(np.float32)
X_test_proc = preprocessor.transform(X_test).astype(np.float32)

y_train_np = y_train.values.astype(np.float32)
y_val_np = y_val.values.astype(np.float32)
y_test_np = y_test.values.astype(np.float32)

input_dim = X_train_proc.shape[1]
print(f'Dimensão de entrada da Rede Neural após pré-processamento: {input_dim} features')

---
## Etapa 4 — Desenvolvimento da Rede Neural Artificial

### Justificativa da Arquitetura
Para dados financeiros tabulares de médio porte (~30 mil amostras, 35 features pós-encoding), arquiteturas convolucionais (CNN) ou recorrentes puras (LSTM) não são ideais por não haver grade espacial ou sequência longa. Optamos por comparar três arquiteturas progressivas em **PyTorch**:

1. **Experimento 1 — MLP Baseline:** Rede densa simples (`64 -> 32 -> 1`) com `ReLU`, sem regularização e `BCEWithLogitsLoss` padrão.
2. **Experimento 2 — MLP Regularizada + Balanceamento:** Rede (`128 -> 64 -> 32 -> 1`) com **`BatchNorm1d`** (estabiliza ativações em features tabulares heterogêneas), **`GELU`**, **`Dropout(0.30)`**, **`AdamW`** (weight decay desacoplado) e **`pos_weight`** na perda para penalizar erros na classe minoritária (inadimplentes).
3. **Experimento 3 — Tabular ResNet (Conexões Residuais):** Inspirada em *Revisiting Deep Learning Models for Tabular Data* (Gorishniy et al., 2021), adiciona **skip connections** (`x + Block(x)`) com ativação `SiLU`, facilitando o fluxo de gradiente e preservando sinais lineares fortes (como `PAY_0`) junto a interações não lineares.

Todos os modelos usam saída linear (*logit*) + `BCEWithLogitsLoss` (numericamente mais estável que `Sigmoid + BCELoss` via *log-sum-exp*).

In [ ]:
# DataLoaders
BATCH_SIZE = 256
train_ds = TensorDataset(torch.from_numpy(X_train_proc), torch.from_numpy(y_train_np))
val_ds = TensorDataset(torch.from_numpy(X_val_proc), torch.from_numpy(y_val_np))
test_ds = TensorDataset(torch.from_numpy(X_test_proc), torch.from_numpy(y_test_np))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

# Peso para classe positiva (negativos / positivos)
pos_weight_val = (y_train_np == 0).sum() / (y_train_np == 1).sum()
print(f'pos_weight calculado no treino: {pos_weight_val:.2f}')

In [ ]:
# Definição das 3 arquiteturas
class BaselineMLP(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


class RegularizedMLP(nn.Module):
    def __init__(self, in_features, dropout=0.30):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


class TabularResBlock(nn.Module):
    def __init__(self, dim, dropout=0.30):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim)
        )

    def forward(self, x):
        return x + self.block(x)


class TabularResNet(nn.Module):
    def __init__(self, in_features, hidden_dim=128, num_blocks=2, dropout=0.30):
        super().__init__()
        self.input_layer = nn.Linear(in_features, hidden_dim)
        self.blocks = nn.Sequential(*[TabularResBlock(hidden_dim, dropout) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        h = self.input_layer(x)
        h = self.blocks(h)
        return self.head(h).squeeze(1)

In [ ]:
# Motor de treinamento com Early Stopping e registro de histórico
def evaluate_loader(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_probs, all_targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(yb.cpu().numpy())
    probs = np.concatenate(all_probs)
    targets = np.concatenate(all_targets)
    avg_loss = total_loss / len(loader.dataset)
    auc = roc_auc_score(targets, probs)
    pr_auc = average_precision_score(targets, probs)
    return avg_loss, auc, pr_auc, probs, targets


def train_model(model, criterion, optimizer, scheduler=None, max_epochs=60, patience=10, name='Modelo'):
    model = model.to(device)
    history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': []}
    best_val_auc = -1.0
    best_state = None
    wait = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            train_loss += loss.item() * xb.size(0)

        train_loss_eval, train_auc, _, _, _ = evaluate_loader(model, train_loader, criterion)
        val_loss, val_auc, _, _, _ = evaluate_loader(model, val_loader, criterion)

        history['train_loss'].append(train_loss_eval)
        history['val_loss'].append(val_loss)
        history['train_auc'].append(train_auc)
        history['val_auc'].append(val_auc)

        if scheduler is not None:
            scheduler.step(val_auc)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'[{name}] Early stopping na época {epoch} | Melhor Val ROC-AUC: {best_val_auc:.4f}')
                break
    else:
        print(f'[{name}] Concluído ({max_epochs} épocas) | Melhor Val ROC-AUC: {best_val_auc:.4f}')

    model.load_state_dict(best_state)
    return model, history

In [ ]:
# Execução dos 3 experimentos
seed_everything(SEED)
pos_w_tensor = torch.tensor([pos_weight_val], dtype=torch.float32, device=device)

# Exp 1: MLP Baseline (sem pos_weight, Adam padrão)
model_base = BaselineMLP(input_dim)
crit_base = nn.BCEWithLogitsLoss()
opt_base = torch.optim.Adam(model_base.parameters(), lr=1e-3)
model_base, hist_base = train_model(model_base, crit_base, opt_base, max_epochs=50, patience=8, name='Exp 1: Baseline MLP')

# Exp 2: MLP Regularizada + pos_weight + AdamW + Scheduler
seed_everything(SEED)
model_reg = RegularizedMLP(input_dim, dropout=0.30)
crit_weighted = nn.BCEWithLogitsLoss(pos_weight=pos_w_tensor)
opt_reg = torch.optim.AdamW(model_reg.parameters(), lr=1e-3, weight_decay=1e-4)
sched_reg = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_reg, mode='max', factor=0.5, patience=4)
model_reg, hist_reg = train_model(model_reg, crit_weighted, opt_reg, sched_reg, max_epochs=60, patience=10, name='Exp 2: Regularized MLP')

# Exp 3: Tabular ResNet + pos_weight + AdamW + Scheduler
seed_everything(SEED)
model_res = TabularResNet(input_dim, hidden_dim=128, num_blocks=2, dropout=0.30)
opt_res = torch.optim.AdamW(model_res.parameters(), lr=1e-3, weight_decay=1e-4)
sched_res = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_res, mode='max', factor=0.5, patience=4)
model_res, hist_res = train_model(model_res, crit_weighted, opt_res, sched_res, max_epochs=60, patience=10, name='Exp 3: Tabular ResNet')

---
## Etapa 5 — Avaliação dos Resultados

Avaliamos:
1. **Curvas de treinamento e validação** (Loss e ROC-AUC) para diagnosticar *overfitting* ou *underfitting*.
2. **Desempenho no conjunto de Teste** (jamais visto durante treino/ajuste).
3. **Comparação tabular dos experimentos** com métricas apropriadas ao desbalanceamento (`ROC-AUC`, `PR-AUC`, `F1`, `Recall`, `Precision`).
4. **Otimização de *threshold* orientada a custo financeiro** para o melhor modelo.

In [ ]:
# 5.1 Curvas de Aprendizado (Loss e ROC-AUC)
histories = {
    'Exp 1: Baseline MLP': hist_base,
    'Exp 2: Regularized MLP': hist_reg,
    'Exp 3: Tabular ResNet': hist_res
}

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, (title, h) in zip(axes, histories.items()):
    epochs = range(1, len(h['train_auc']) + 1)
    ax.plot(epochs, h['train_auc'], label='Treino ROC-AUC', linewidth=2)
    ax.plot(epochs, h['val_auc'], label='Validação ROC-AUC', linewidth=2, linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Época')
    ax.set_ylabel('ROC-AUC')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, (title, h) in zip(axes, histories.items()):
    epochs = range(1, len(h['train_loss']) + 1)
    ax.plot(epochs, h['train_loss'], label='Treino Loss', linewidth=2)
    ax.plot(epochs, h['val_loss'], label='Validação Loss', linewidth=2, linestyle='--')
    ax.set_title(f'{title} — Curva de Perda')
    ax.set_xlabel('Época')
    ax.set_ylabel('BCE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Avaliação comparativa no conjunto de Teste
models_eval = [
    ('Exp 1: Baseline MLP', model_base, crit_base),
    ('Exp 2: Regularized MLP', model_reg, crit_weighted),
    ('Exp 3: Tabular ResNet', model_res, crit_weighted),
]

results = []
test_predictions = {}

for name, m, crit in models_eval:
    _, auc, pr_auc, probs, targets = evaluate_loader(m, test_loader, crit)
    preds = (probs >= 0.5).astype(int)
    results.append({
        'Modelo': name,
        'ROC-AUC': round(auc, 4),
        'PR-AUC': round(pr_auc, 4),
        'F1-Score (0.5)': round(f1_score(targets, preds), 4),
        'Recall (0.5)': round(recall_score(targets, preds), 4),
        'Precision (0.5)': round(precision_score(targets, preds), 4)
    })
    test_predictions[name] = (probs, targets)

df_results = pd.DataFrame(results)
display(df_results)

In [ ]:
# 5.3 Curvas ROC e Precision-Recall no Teste
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for name, (probs, targets) in test_predictions.items():
    fpr, tpr, _ = roc_curve(targets, probs)
    axes[0].plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={roc_auc_score(targets, probs):.3f})')

    prec, rec, _ = precision_recall_curve(targets, probs)
    axes[1].plot(rec, prec, linewidth=2, label=f'{name} (PR-AUC={average_precision_score(targets, probs):.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title('Curva ROC — Conjunto de Teste')
axes[0].set_xlabel('Taxa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos (Recall)')
axes[0].legend()

axes[1].set_title('Curva Precision-Recall — Conjunto de Teste')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Otimização de Threshold sob Ótica de Custo Financeiro (selecionando melhor modelo por ROC-AUC)
best_name = df_results.sort_values('ROC-AUC', ascending=False).iloc[0]['Modelo']
best_model = dict([(n, m) for n, m, _ in models_eval])[best_name]
best_crit = dict([(n, c) for n, _, c in models_eval])[best_name]

# Calibra o threshold na VALIDAÇÃO (nunca no teste!) e aplica no TESTE
_, _, _, val_probs, val_targets = evaluate_loader(best_model, val_loader, best_crit)
test_probs, test_targets = test_predictions[best_name]

COST_FN = 5.0  # Custo relativo de aprovar um inadimplente
COST_FP = 1.0  # Custo relativo de recusar um bom pagador

thresholds = np.linspace(0.10, 0.90, 81)
val_costs = []
for t in thresholds:
    preds_t = (val_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(val_targets, preds_t).ravel()
    val_costs.append(fn * COST_FN + fp * COST_FP)

opt_idx = int(np.argmin(val_costs))
opt_threshold = thresholds[opt_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(thresholds, val_costs, linewidth=2, color='#805ad5')
axes[0].axvline(opt_threshold, color='red', linestyle='--', label=f'Threshold ótimo = {opt_threshold:.2f}')
axes[0].axvline(0.50, color='gray', linestyle=':', label='Threshold padrão = 0.50')
axes[0].set_title(f'Custo Financeiro Esperado na Validação ({best_name})')
axes[0].set_xlabel('Threshold de Decisão')
axes[0].set_ylabel('Custo Total Relativo (5*FN + 1*FP)')
axes[0].legend()

cm_opt = confusion_matrix(test_targets, (test_probs >= opt_threshold).astype(int))
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Adimplente', 'Inadimplente'], yticklabels=['Adimplente', 'Inadimplente'])
axes[1].set_title(f'Matriz de Confusão no Teste (Threshold={opt_threshold:.2f})')
axes[1].set_xlabel('Previsto')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

print(f'Relatório no Teste com Threshold Otimizado ({opt_threshold:.2f}):')
print(classification_report(test_targets, (test_probs >= opt_threshold).astype(int), target_names=['Adimplente', 'Inadimplente']))

---
## Etapa 6 — Conclusão

### 6.1 Principais Resultados Obtidos
- A inclusão de **regularização (`BatchNorm1d`, `Dropout`, `AdamW`)**, **tratamento de desbalanceamento (`pos_weight`)** e **conexões residuais (`TabularResNet`)** reduziu o *overfitting* observado na MLP baseline e elevou o **Recall** e o **F1-Score** sem sacrificar o poder de ordenação (ROC-AUC / PR-AUC).
- A calibração do *threshold* baseada em **custo assimétrico (FN=5x FP)** na validação demonstrou como transformar probabilidades da rede neural em uma política de crédito acionável.

### 6.2 Atendimento ao Objetivo e Dificuldades Encontradas
- **Objetivo atendido:** Construímos um pipeline completo, reprodutível e livre de *data leakage* para estimar risco de inadimplência.
- **Dificuldades encontradas:**
  1. Códigos categóricos não documentados em `EDUCATION` e `MARRIAGE` que poderiam gerar ruído no *One-Hot Encoding*.
  2. Outliers extremos em `BILL_AMT` e `PAY_AMT`, contornados com `RobustScaler` e transformações logarítmicas.
  3. Sensibilidade da MLP baseline ao desbalanceamento (tende a subestimar inadimplentes sem `pos_weight`).

### 6.3 Limitações e Melhorias Futuras
- **Limitações:** Dados históricos de 2005 (Taiwan) não refletem choques macroeconômicos atuais nem trazem variáveis de renda declarada ou *score* de birô externo; além disso, redes neurais têm menor interpretabilidade intrínseca que modelos lineares.
- **Melhorias futuras:**
  1. Aplicar **SHAP (*SHapley Additive exPlanations*)** para explicabilidade individual das decisões de crédito (requisito regulatório).
  2. Testar **Entity Embeddings** para variáveis categóricas e arquiteturas baseadas em atenção (*FT-Transformer*).
  3. Calibração pós-treino das probabilidades (*Platt Scaling* ou *Isotonic Regression*) quando se usa `pos_weight`.